# H3 · DINOv2 backbone comparison at 336 pixels

Compare **frozen ViT-S/14, ViT-B/14, and ViT-L/14** using original training images
plus four augmented views, a newly trained MLP-BN-Dropout head, and seeds 123/2026.
Each representation gets its own training-only normalization. No external labeled
data, pseudo-labeling, test-time augmentation, or ensemble is used.

**Colab:** select an A100 GPU, edit section 1, then Run all. This notebook contains
no measured H3 results. It does not require the lost F0 checkpoint. If an F0 checkpoint
is found, evaluate it separately at its original 224-pixel resolution; otherwise
compare against the explicitly labeled previously reported F0 reference.

The original F0 procedure was inspected at repository commit
`5a393fa56ae33a183a2268253528a795b5a450fe`. All existing experiments are preserved.
Changed resolution and controlled data-loader RNG mean this is a new experiment,
not a promise to replay historical F0 augmentation draws exactly.

## 0. Environment
Keep Colab's compatible PyTorch/TorchVision installation. Frozen features are extracted
in **FP32 for all candidates**; TF32 is disabled for a controlled comparison. Only the
small head is trained. One backbone resides on the GPU at a time. Batch sizes below
are conservative for an A100; reduce the affected candidate's batch size on an OOM.

In [ ]:
%pip -q install "timm==1.0.22" "scikit-learn==1.7.2" "pandas==2.2.3" "Pillow==11.3.0" "tqdm==4.67.1" "matplotlib==3.10.7" "kagglehub==1.0.3"

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import random
import uuid
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| TorchVision:', torchvision.__version__, '| Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    warnings.warn('CPU mode supports functional checks; real ViT-B/L feature extraction will be slow.')

## 1. Configuration and storage
Drive is enabled by default. Each execution creates a new run folder under
`MyDrive/Car-Classification-Benchmark/student_outputs_H3/`. Frozen-feature caches are
shared by content signature; checkpoints/results always use separate run folders.

For Kaggle download, enable Colab Secret `KAGGLE_API_TOKEN` and accept the original
competition rules. Alternatively disable downloading and point `DATA_ROOT` to extracted
`train/`, `val/`, `test/`, and `sample_submission.csv`.

`F0_CHECKPOINT=None` enables optional discovery in the existing F0/recovery folders.
Set a specific path if desired; an explicitly supplied missing checkpoint is an error.
H3 runs even if no F0 checkpoint is available. Test images are used for inference only.
Set `SAVE_TEST_PROBABILITIES=False` to postpone test inference, but it is required for
automatic submission generation. This experiment does not implement 448-pixel runs.

In [ ]:
USE_DRIVE = True
DOWNLOAD_FROM_KAGGLE = True
DATA_ROOT = Path('/content/car_data')
ROOT = Path('/content/drive/MyDrive/Car-Classification-Benchmark') if USE_DRIVE else Path('/content')
OUTPUT_ROOT = ROOT / 'student_outputs_H3'
F0_CHECKPOINT = None
AUTO_FIND_F0 = True
SAVE_TEST_PROBABILITIES = True
GENERATE_SUBMISSION = True
REUSE_FEATURE_CACHE = True

CANDIDATES = [
    {'name': 'dinov2_vits14', 'dim': 384, 'batch_size': 32},
    {'name': 'dinov2_vitb14', 'dim': 768, 'batch_size': 16},
    {'name': 'dinov2_vitl14', 'dim': 1024, 'batch_size': 8},
]
HUB_REF = '7764ea0f912e53c92e82eb78a2a1631e92725fc8'
IMAGE_SIZE = 336
RESIZE_SIZE = 384
NUM_CLASSES = 20
NUM_WORKERS = 0  # Avoid Colab multiprocessing iterator-cleanup failures.
SEED_AUG = 123
SEED_HEAD = 2026
N_AUG = 4
HEAD_BATCH_SIZE = 64
HEAD_EPOCHS = 35
HEAD_PATIENCE = 7
HEAD_LR = 1e-3
HEAD_WEIGHT_DECAY = 1e-3
HEAD_CONFIG = {'hidden_dim': 128, 'use_bn': True, 'dropout': 0.3}
# These are the user's measured earlier results, not results computed in this notebook.
REPORTED_F0 = {'val_accuracy': 0.98, 'val_macro_f1': 0.9802785950383205,
               'val_loss': 0.17501021441072226}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(SEED_AUG)
assert IMAGE_SIZE % 14 == 0 and RESIZE_SIZE >= IMAGE_SIZE
assert len({c['name'] for c in CANDIDATES}) == len(CANDIDATES) > 0
assert NUM_WORKERS == 0, 'This controlled comparison uses single-process image loading.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    DATA_ROOT = Path(kagglehub.competition_download('idnn-swe-3052-assignment-3-flowers-classification'))
for folder in ('train', 'val', 'test'):
    assert (DATA_ROOT / folder).is_dir(), f'Missing {DATA_ROOT / folder}; edit DATA_ROOT.'
assert (DATA_ROOT / 'sample_submission.csv').is_file(), 'Original sample_submission.csv required.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
CACHE_DIR = OUTPUT_ROOT / 'feature_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ENVIRONMENT = {'python_packages': {p: importlib.metadata.version(p) for p in
               ['torch', 'torchvision', 'timm', 'scikit-learn', 'pandas', 'Pillow', 'kagglehub']},
               'device': str(DEVICE), 'cuda': str(torch.version.cuda),
               'gpu': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None}
print('Data:', DATA_ROOT, '\nNew run:', OUTPUT_DIR)

## 2. Original geometry augmentation at higher resolution
Train: RandomResizedCrop(336), scale (0.8,1.0), ratio (0.9,1.1), horizontal flip 0.5.
Validation/test: Resize(384), CenterCrop(336). All resizing is BICUBIC with original
ImageNet normalization. The original-to-crop scale ratio 256/224 is preserved.
Class-folder prefixes are checked against competition labels. Validation labels are
never used to train the head or fit normalization.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def make_eval_transform(image_size, resize_size):
    return transforms.Compose([
        transforms.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(image_size), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1),
                                 interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_transform = make_eval_transform(IMAGE_SIZE, RESIZE_SIZE)
train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=train_transform)
train_original_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=eval_transform)
val_ds = datasets.ImageFolder(DATA_ROOT / 'val', transform=eval_transform)
assert train_ds.samples == train_original_ds.samples
assert train_ds.class_to_idx == val_ds.class_to_idx
assert len(train_ds.classes) == NUM_CLASSES
assert (len(train_ds), len(val_ds)) == (700, 200), 'Expected original 700/200 split.'
for label, folder in enumerate(train_ds.classes):
    assert folder.startswith(f'{label:03d}_'), f'Class label mismatch: {folder}'
assert not ({Path(p).resolve() for p, _ in train_ds.samples}
            & {Path(p).resolve() for p, _ in val_ds.samples})
CLASS_TO_IDX = dict(train_ds.class_to_idx)
VAL_IDS = [Path(p).relative_to(DATA_ROOT / 'val').as_posix() for p, _ in val_ds.samples]
Y_VAL = np.asarray(val_ds.targets, dtype=np.int64)
PREPROCESSING = {'image_size': IMAGE_SIZE, 'resize_size': RESIZE_SIZE,
                 'train': repr(train_transform), 'eval': repr(eval_transform),
                 'mean': MEAN, 'std': STD, 'interpolation': 'BICUBIC',
                 'n_aug': N_AUG, 'include_original': True}
print('Training:', len(train_ds), '| Validation:', len(val_ds), '| Classes:', NUM_CLASSES)

## 3. Test IDs and dataset fingerprint
Test order comes only from the original sample CSV. IDs remain strings, including
leading zeros. Dataset content hashes prevent cache reuse after images or labels
change. Hashes are recorded for reproducibility; images and weights are not added to Git.

In [ ]:
def discover_test_images(test_dir, sample_path):
    sample = pd.read_csv(sample_path, dtype={'image_id': str}, keep_default_na=False)
    assert list(sample.columns) == ['image_id', 'label']
    ids = sample.image_id.tolist()
    assert ids and all(isinstance(x, str) and x.strip() for x in ids)
    assert len(ids) == len(set(ids)), 'Duplicate sample image IDs.'
    extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    files = sorted(p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in extensions)
    by_id = {p.stem: p for p in files}
    assert len(by_id) == len(files), 'Duplicate test filename stems.'
    missing, extra = set(ids) - set(by_id), set(by_id) - set(ids)
    assert not missing and not extra, f'Test ID mismatch: missing={sorted(missing)[:5]}, extra={sorted(extra)[:5]}'
    return sample, ids, by_id

class TestImages(Dataset):
    def __init__(self, ids, by_id, transform):
        self.ids, self.by_id, self.transform = list(ids), by_id, transform
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, index):
        with Image.open(self.by_id[self.ids[index]]) as im:
            return self.transform(im.convert('RGB'))

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

sample_path = DATA_ROOT / 'sample_submission.csv'
sample, TEST_IDS, test_by_id = discover_test_images(DATA_ROOT / 'test', sample_path)
test_ds = TestImages(TEST_IDS, test_by_id, eval_transform)
manifest = []
for split, image_samples in [('train', train_ds.samples), ('val', val_ds.samples),
                              ('test', [(test_by_id[i], None) for i in TEST_IDS])]:
    for path, label in tqdm(image_samples, desc=f'Fingerprint {split}'):
        manifest.append({'path': Path(path).relative_to(DATA_ROOT).as_posix(),
                         'label': label, 'sha256': file_sha256(path)})
DATA_SIGNATURE = hashlib.sha256(json.dumps({'images': manifest, 'test_ids': TEST_IDS,
                         'classes': CLASS_TO_IDX}, sort_keys=True).encode()).hexdigest()
(OUTPUT_DIR / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2))
print('Test images:', len(TEST_IDS), '| First IDs:', TEST_IDS[:3])

## 4. Frozen feature extraction and validated caches
Reset seed 123 after loading each backbone. Dedicated DataLoader generators keep
iterator creation from consuming crop RNG, so candidates see the same four augmented
views despite different extraction batch sizes. Validation/test receive one view.
Keep feature tensors on CPU. Each cache records model, source revision, precision,
preprocessing, dataset signature, and software versions.

In [ ]:
def make_image_loader(dataset, batch_size):
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
                      pin_memory=(DEVICE.type == 'cuda'),
                      generator=torch.Generator().manual_seed(SEED_AUG))

def load_backbone(candidate):
    backbone = torch.hub.load(f'facebookresearch/dinov2:{HUB_REF}', candidate['name'],
                              pretrained=True, trust_repo=True, skip_validation=True).to(DEVICE)
    assert backbone.embed_dim == candidate['dim']
    backbone.requires_grad_(False)
    backbone.eval()
    return backbone

@torch.inference_mode()
def extract_features(backbone, dataset, batch_size, feature_dim):
    backbone.eval()
    parts = []
    for batch in tqdm(make_image_loader(dataset, batch_size), desc='Frozen features', leave=False):
        images = batch[0] if isinstance(batch, (tuple, list)) else batch
        features = backbone(images.to(DEVICE, non_blocking=True)).float()
        assert features.shape == (images.size(0), feature_dim)
        assert torch.isfinite(features).all(), 'Non-finite backbone output.'
        parts.append(features.cpu())
    return torch.cat(parts, dim=0)

def cache_metadata(candidate):
    return {'schema': 1, 'backbone': candidate['name'], 'feature_dim': candidate['dim'],
            'hub_ref': HUB_REF, 'dataset_signature': DATA_SIGNATURE,
            'preprocessing': PREPROCESSING, 'seed_aug': SEED_AUG, 'precision': 'float32',
            'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
            'device_type': DEVICE.type, 'gpu': ENVIRONMENT['gpu'], 'cuda': str(torch.version.cuda),
            'extraction_batch_size': candidate['batch_size'], 'num_workers': NUM_WORKERS}

def validate_feature_cache(cached, metadata):
    assert cached['metadata'] == metadata, 'Feature cache metadata mismatch.'
    expected = {'X_train': len(train_ds) * (N_AUG + 1), 'X_val': len(val_ds)}
    for key, rows in expected.items():
        value = cached[key]
        assert value.dtype == torch.float32 and value.shape == (rows, metadata['feature_dim'])
        assert torch.isfinite(value).all(), f'Invalid {key} in feature cache.'

def get_features(backbone, candidate):
    metadata = cache_metadata(candidate)
    signature = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()
    path = CACHE_DIR / f'{candidate["name"]}_{signature}.pt'
    if REUSE_FEATURE_CACHE and path.is_file():
        cached = torch.load(path, map_location='cpu', weights_only=True)
        validate_feature_cache(cached, metadata)
        print('Validated feature-cache hit:', path.name)
        return cached, signature
    seed_everything(SEED_AUG)
    views = [extract_features(backbone, train_original_ds, candidate['batch_size'], candidate['dim'])]
    for view in range(N_AUG):
        print(f'Augmented training view {view + 1}/{N_AUG}')
        views.append(extract_features(backbone, train_ds, candidate['batch_size'], candidate['dim']))
    cached = {'metadata': metadata, 'X_train': torch.cat(views, dim=0),
              'X_val': extract_features(backbone, val_ds, candidate['batch_size'], candidate['dim'])}
    validate_feature_cache(cached, metadata)
    if not path.exists():
        temporary = path.with_suffix('.tmp')
        torch.save(cached, temporary)
        temporary.replace(path)
    return cached, signature

## 5. Train a fresh, compatible head for each representation
The head is Linear(dim,128) → BN → ReLU → Dropout(0.3) → Linear(128,20).
Fresh head BatchNorm **does** learn its running statistics, as in F0; H2's fixed-BN
policy applied to a previously trained head only. Train only on original + augmented
training features. Labels repeat once per complete view, not once per image.

AdamW: LR 1e-3, weight decay 1e-3, batch 64, max 35 epochs, patience 7, seed 2026.
Selection is accuracy then macro-F1 (F0 originally used accuracy alone). Save every
head epoch checkpoint and validation probabilities, plus a separate lowest-loss
checkpoint, so useful candidates are never lost. Lower loss alone is not an accuracy gain.

In [ ]:
def build_head(feature_dim):
    return nn.Sequential(nn.Linear(feature_dim, 128), nn.BatchNorm1d(128), nn.ReLU(),
                         nn.Dropout(0.3), nn.Linear(128, NUM_CLASSES))

def fit_normalization(X_train):
    assert X_train.ndim == 2 and torch.isfinite(X_train).all()
    return X_train.mean(0), X_train.std(0, unbiased=False).clamp_min(1e-6)

def metric_rank(metrics):
    return metrics['val_accuracy'], metrics['val_macro_f1']

def improves_reference(metrics, reference):
    # Ignore floating-point noise when comparing against a historical sklearn result.
    accuracy_delta = metrics['val_accuracy'] - reference['val_accuracy']
    if accuracy_delta > 1e-12:
        return True
    return abs(accuracy_delta) <= 1e-12 and metrics['val_macro_f1'] > reference['val_macro_f1'] + 1e-12

def compute_metrics(logits, targets):
    assert logits.shape == (len(targets), NUM_CLASSES) and torch.isfinite(logits).all()
    predictions = logits.argmax(1).cpu().numpy()
    return {'val_loss': float(nn.functional.cross_entropy(logits, targets).item()),
            'val_accuracy': float(accuracy_score(targets.cpu().numpy(), predictions)),
            'val_macro_f1': float(f1_score(targets.cpu().numpy(), predictions,
                                         labels=list(range(NUM_CLASSES)), average='macro', zero_division=0))}

@torch.inference_mode()
def head_logits(head, X, mean, std):
    head.eval()
    parts = []
    for features in X.split(256):
        normalized = (features - mean) / std
        parts.append(head(normalized.to(DEVICE)).float().cpu())
    return torch.cat(parts)

def cpu_state(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

def save_head_checkpoint(path, head, mean, std, candidate, signature, epoch, metrics):
    torch.save({'head_state_dict': cpu_state(head), 'feature_mean': mean.cpu(),
                'feature_std': std.cpu(), 'class_to_idx': CLASS_TO_IDX,
                'backbone': candidate['name'], 'feature_dim': candidate['dim'], 'hub_ref': HUB_REF,
                'preprocessing': PREPROCESSING, 'head_experiment': 'E4_mlp_bn_drop03',
                'head_config': HEAD_CONFIG, 'seed_aug': SEED_AUG, 'seed_head': SEED_HEAD,
                'epoch': epoch, 'validation_metrics': dict(metrics), 'feature_cache_signature': signature,
                'selection_rule': 'accuracy_then_macro_f1', 'backbone_frozen': True,
                'training': {'batch_size': HEAD_BATCH_SIZE, 'max_epochs': HEAD_EPOCHS,
                             'patience': HEAD_PATIENCE, 'optimizer': 'AdamW', 'lr': HEAD_LR,
                             'weight_decay': HEAD_WEIGHT_DECAY}, 'environment': ENVIRONMENT}, path)

def train_head(cached, candidate, signature, run_dir):
    X_train, X_val = cached['X_train'], cached['X_val']
    mean, std = fit_normalization(X_train)
    Z_train = (X_train - mean) / std
    y_train = torch.tensor(train_ds.targets, dtype=torch.long).repeat(N_AUG + 1)
    y_val = torch.tensor(val_ds.targets, dtype=torch.long)
    assert len(Z_train) == len(y_train) and len(y_train) % HEAD_BATCH_SIZE != 1
    seed_everything(SEED_HEAD)
    head = build_head(candidate['dim']).to(DEVICE)
    loader = DataLoader(TensorDataset(Z_train, y_train), batch_size=HEAD_BATCH_SIZE, shuffle=True,
                        num_workers=0, generator=torch.Generator().manual_seed(SEED_HEAD))
    optimizer = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    history, bad_epochs = [], 0
    best_metrics, lowest_loss = None, math.inf
    epoch_dir = run_dir / 'epochs'
    epoch_dir.mkdir(exist_ok=False)
    for epoch in range(1, HEAD_EPOCHS + 1):
        head.train()
        loss_sum = 0.0
        for features, labels in loader:
            features, labels = features.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(head(features), labels)
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite head loss; inspect extracted features.')
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * labels.size(0)
        logits = head_logits(head, X_val, mean, std)
        metrics = compute_metrics(logits, y_val)
        history.append({'epoch': epoch, 'train_loss': loss_sum / len(y_train), **metrics,
                        'head_lr': optimizer.param_groups[0]['lr']})
        checkpoint_path = epoch_dir / f'epoch_{epoch:02d}.pt'
        save_head_checkpoint(checkpoint_path, head, mean, std, candidate, signature, epoch, metrics)
        np.savez_compressed(epoch_dir / f'epoch_{epoch:02d}_val_probs.npz',
                            image_id=np.asarray(VAL_IDS), target=Y_VAL,
                            probabilities=logits.softmax(1).numpy(),
                            class_names=np.asarray(train_ds.classes))
        pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
        print(f'{candidate["name"]} | epoch {epoch:02d} | train {history[-1]["train_loss"]:.4f} '
              f'| val {metrics["val_loss"]:.4f} | accuracy {metrics["val_accuracy"]:.2%} '
              f'| macro-F1 {metrics["val_macro_f1"]:.4f}')
        if metrics['val_loss'] < lowest_loss:
            lowest_loss = metrics['val_loss']
            save_head_checkpoint(run_dir / 'lowest_loss_head.pt', head, mean, std,
                                 candidate, signature, epoch, metrics)
        if best_metrics is None or metric_rank(metrics) > metric_rank(best_metrics):
            best_metrics, bad_epochs = dict(metrics), 0
            save_head_checkpoint(run_dir / 'best_head.pt', head, mean, std,
                                 candidate, signature, epoch, metrics)
        else:
            bad_epochs += 1
            if bad_epochs >= HEAD_PATIENCE:
                print('Early stopping: no accuracy/F1 ranking improvement.')
                break
    best = torch.load(run_dir / 'best_head.pt', map_location='cpu', weights_only=True)
    head.load_state_dict(best['head_state_dict'], strict=True)
    head.eval()
    logits = head_logits(head, X_val, mean, std)
    restored_metrics = compute_metrics(logits, y_val)
    for key, value in best['validation_metrics'].items():
        assert np.isclose(restored_metrics[key], value, rtol=0, atol=1e-6), f'Restore mismatch: {key}'
    return head, mean, std, best['epoch'], restored_metrics, logits.softmax(1).numpy(), history

## 6. Run S/14, B/14, and L/14 sequentially
All checkpoints, per-epoch probabilities, and feature-cache references are preserved.
The validation CSV uses relative class/image paths because bare validation stems can
repeat across class folders. Probability columns always follow class indices 0–19.
No backbone parameters are updated. Test probabilities contain no test labels.

In [ ]:
def save_predictions(run_dir, prefix, ids, probabilities, targets=None):
    probabilities = np.asarray(probabilities, dtype=np.float32)
    assert probabilities.shape == (len(ids), NUM_CLASSES) and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(1), 1, atol=1e-5)
    assert len(set(ids)) == len(ids)
    records = pd.DataFrame({'image_id': ids, 'prediction': probabilities.argmax(1),
                            'confidence': probabilities.max(1)})
    if targets is not None:
        records['target'] = targets
        records['correct'] = records.prediction.to_numpy() == np.asarray(targets)
    for label in range(NUM_CLASSES):
        records[f'p_{label:02d}'] = probabilities[:, label]
    records.to_csv(run_dir / f'{prefix}_predictions.csv', index=False)
    payload = {'image_id': np.asarray(ids), 'probabilities': probabilities,
               'class_names': np.asarray(train_ds.classes)}
    if targets is not None:
        payload['target'] = np.asarray(targets, dtype=np.int64)
    np.savez_compressed(run_dir / f'{prefix}_probabilities.npz', **payload)

RUN_CONFIG = {'candidates': CANDIDATES, 'preprocessing': PREPROCESSING,
              'seed_aug': SEED_AUG, 'seed_head': SEED_HEAD, 'hub_ref': HUB_REF,
              'dataset_signature': DATA_SIGNATURE, 'class_to_idx': CLASS_TO_IDX,
              'head_config': HEAD_CONFIG, 'head_epochs': HEAD_EPOCHS,
              'head_lr': HEAD_LR, 'head_weight_decay': HEAD_WEIGHT_DECAY,
              'head_batch_size': HEAD_BATCH_SIZE, 'head_patience': HEAD_PATIENCE,
              'environment': ENVIRONMENT}
(OUTPUT_DIR / 'H3_run_config.json').write_text(json.dumps(RUN_CONFIG, indent=2))
results, validation_probabilities, test_probabilities, histories = [], {}, {}, {}
for candidate in CANDIDATES:
    print('\nStarting:', candidate)
    run_dir = OUTPUT_DIR / candidate['name']
    run_dir.mkdir(exist_ok=False)
    backbone = None
    head = None
    try:
        backbone = load_backbone(candidate)
        cached, signature = get_features(backbone, candidate)
        head, mean, std, best_epoch, metrics, val_probs, history = train_head(
            cached, candidate, signature, run_dir)
        assert not backbone.training and all(not p.requires_grad and p.grad is None
                                              for p in backbone.parameters())
        save_predictions(run_dir, 'val', VAL_IDS, val_probs, Y_VAL)
        pd.DataFrame(classification_report(Y_VAL, val_probs.argmax(1), labels=list(range(NUM_CLASSES)),
                     target_names=train_ds.classes, zero_division=0, output_dict=True)).T.to_csv(
                         run_dir / 'classification_report.csv')
        validation_probabilities[candidate['name']] = val_probs
        histories[candidate['name']] = history
        if SAVE_TEST_PROBABILITIES:
            X_test = extract_features(backbone, test_ds, candidate['batch_size'], candidate['dim'])
            probabilities = head_logits(head, X_test, mean, std).softmax(1).numpy()
            save_predictions(run_dir, 'test', TEST_IDS, probabilities)
            test_probabilities[candidate['name']] = probabilities
            del X_test
        result = {'model': candidate['name'], 'image_size': IMAGE_SIZE,
                  'feature_dim': candidate['dim'], 'best_epoch': best_epoch, **metrics,
                  'head_parameters': sum(p.numel() for p in head.parameters()),
                  'checkpoint': str(run_dir / 'best_head.pt')}
        results.append(result)
        (run_dir / 'summary.json').write_text(json.dumps(result, indent=2))
        pd.DataFrame(results).to_csv(OUTPUT_DIR / 'H3_comparison.csv', index=False)
        del cached, head, mean, std
        head = None
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError(f'GPU OOM for {candidate["name"]}. Reduce its batch_size in section 1 '
                           'and rerun in a new run folder; completed candidate files remain saved.') from exc
    finally:
        del backbone, head
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
comparison = pd.DataFrame(results).sort_values(['val_accuracy', 'val_macro_f1'],
                                               ascending=False, kind='stable').reset_index(drop=True)
display(comparison)

## 7. Optional actual F0 baseline evaluation
If found, F0 uses its own saved normalization and original Resize(256)/CenterCrop(224).
No normalization is refitted. Missing optional F0 weights never prevent H3 comparison.
An explicit invalid checkpoint is reported as an error. The historical reference is
clearly distinguished from a newly measured baseline in the saved summary.

In [ ]:
def resolve_f0_checkpoint():
    if F0_CHECKPOINT is not None:
        path = Path(F0_CHECKPOINT)
        if not path.is_file():
            raise FileNotFoundError(f'Explicit F0_CHECKPOINT does not exist: {path}; set None to skip.')
        return path
    if not AUTO_FIND_F0:
        return None
    matches = []
    for folder in [ROOT / 'student_outputs', ROOT / 'student_outputs_F0_recovery']:
        if folder.is_dir():
            matches.extend(folder.rglob('dinov2_p3_e4_aug123_head2026.pt'))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def evaluate_f0_reference(path):
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    expected = {'backbone': 'dinov2_vits14', 'preprocessing': 'P3_geometry_multiview',
                'head_experiment': 'E4_mlp_bn_drop03', 'head_config': HEAD_CONFIG,
                'seed_aug': 123, 'seed_head': 2026, 'class_to_idx': CLASS_TO_IDX}
    for key, value in expected.items():
        assert checkpoint[key] == value, f'F0 checkpoint metadata mismatch: {key}'
    mean, std = checkpoint['feature_mean'].reshape(-1), checkpoint['feature_std'].reshape(-1)
    assert mean.numel() == std.numel() == 384
    assert torch.isfinite(mean).all() and torch.isfinite(std).all() and (std > 0).all()
    backbone = load_backbone({'name': 'dinov2_vits14', 'dim': 384})
    head = None
    try:
        dataset = datasets.ImageFolder(DATA_ROOT / 'val', transform=make_eval_transform(224, 256))
        assert dataset.samples == val_ds.samples and dataset.class_to_idx == CLASS_TO_IDX
        X_val = extract_features(backbone, dataset, 32, 384)
        head = build_head(384).to(DEVICE)
        head.load_state_dict(checkpoint['head_state_dict'], strict=True)
        logits = head_logits(head, X_val, mean, std)
        metrics = compute_metrics(logits, torch.tensor(Y_VAL, dtype=torch.long))
        probabilities = logits.softmax(1).numpy()
        folder = OUTPUT_DIR / 'F0_reference'
        folder.mkdir(exist_ok=False)
        save_predictions(folder, 'val', VAL_IDS, probabilities, Y_VAL)
        return metrics, probabilities
    finally:
        del backbone, head
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

reference_path = resolve_f0_checkpoint()
if reference_path is not None:
    print('Evaluating recovered F0 checkpoint:', reference_path)
    try:
        reference_metrics, f0_probabilities = evaluate_f0_reference(reference_path)
    except Exception as exc:
        raise RuntimeError('F0 comparison failed. H3 candidate artifacts are already saved. '
                           'Check the F0 checkpoint, or set F0_CHECKPOINT=None and AUTO_FIND_F0=False '
                           'and rerun section 7 onward to use the reported reference.') from exc
    reference_kind = 'measured_from_checkpoint'
    validation_probabilities['F0_reference'] = f0_probabilities
else:
    reference_metrics = dict(REPORTED_F0)
    reference_kind = 'previously_reported_not_reevaluated'
    print('No F0 checkpoint found. Using previously reported metrics as reference only.')
print('Reference:', reference_kind, reference_metrics)

## 8. Comparison, disagreements, plots, and error gallery
A disagreement table reveals whether models correct different errors. It does not
perform ensembling. One validation image is 0.5 percentage points, so a small gain on
this split needs confirmation rather than a claim of guaranteed Kaggle improvement.

In [ ]:
comparison['accuracy_delta_pp'] = 100 * (comparison.val_accuracy - reference_metrics['val_accuracy'])
comparison['macro_f1_delta_pp'] = 100 * (comparison.val_macro_f1 - reference_metrics['val_macro_f1'])
comparison.to_csv(OUTPUT_DIR / 'H3_comparison.csv', index=False)
display(comparison)
names = list(validation_probabilities)
disagreements = []
for a_index, a in enumerate(names):
    for b in names[a_index + 1:]:
        a_pred, b_pred = validation_probabilities[a].argmax(1), validation_probabilities[b].argmax(1)
        a_correct, b_correct = a_pred == Y_VAL, b_pred == Y_VAL
        disagreements.append({'model_a': a, 'model_b': b,
                              'only_a_correct': int((a_correct & ~b_correct).sum()),
                              'only_b_correct': int((~a_correct & b_correct).sum()),
                              'both_wrong': int((~a_correct & ~b_correct).sum()),
                              'both_correct': int((a_correct & b_correct).sum()),
                              'different_predictions': int((a_pred != b_pred).sum())})
pd.DataFrame(disagreements).to_csv(OUTPUT_DIR / 'H3_disagreements.csv', index=False)
display(pd.DataFrame(disagreements))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, history in histories.items():
    frame = pd.DataFrame(history)
    for ax, metric in zip(axes, ['train_loss', 'val_loss', 'val_accuracy']):
        ax.plot(frame.epoch, frame[metric] * (100 if metric == 'val_accuracy' else 1), label=name)
for ax, title in zip(axes, ['Training loss', 'Validation loss', 'Validation accuracy (%)']):
    ax.set(xlabel='Head training epoch', ylabel=title, title=title)
    ax.grid(alpha=.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'H3_training_curves.png', dpi=160)
plt.show()

best_name = str(comparison.iloc[0]['model'])
best_probs = validation_probabilities[best_name]
wrong = np.flatnonzero(best_probs.argmax(1) != Y_VAL)
if len(wrong):
    rows = math.ceil(min(len(wrong), 12) / 3)
    fig, axes = plt.subplots(rows, 3, figsize=(13, 3.5 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, index in zip(axes.flat, wrong[:12]):
        with Image.open(val_ds.samples[index][0]) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f'True: {train_ds.classes[Y_VAL[index]]}\n'
                     f'Pred: {train_ds.classes[best_probs[index].argmax()]} '
                     f'({best_probs[index].max():.1%})', fontsize=9)
    fig.suptitle(f'{best_name}: validation errors (up to 12)')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'H3_validation_errors.png', dpi=150)
    plt.show()

## 9. Submission gate and summary
Generate one separate H3 CSV only if the selected model beats the F0 reference by
accuracy first, macro-F1 second. If F0 was not reevaluated, this comparison uses the
previously reported reference and is explicitly labeled. Test probability files can
still be retained for later analysis when no submission qualifies. There is no
automatic Kaggle upload. Checkpoints contain head weights/statistics and enough
metadata to reload the pinned original frozen backbone; full backbone weights are
downloaded again when needed.

In [ ]:
def build_submission(ids, probabilities, original_sample):
    assert probabilities.shape == (len(ids), NUM_CLASSES)
    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(1), 1, atol=1e-5)
    labels = probabilities.argmax(1).astype(np.int64)
    submission = pd.DataFrame({'image_id': ids, 'label': labels})
    assert list(submission.columns) == ['image_id', 'label']
    assert len(submission) == len(original_sample) and submission.image_id.is_unique
    assert submission.image_id.tolist() == original_sample.image_id.tolist()
    assert submission.label.between(0, NUM_CLASSES - 1).all()
    return submission

best_result = comparison.iloc[0].to_dict()
promising_candidate = improves_reference(best_result, reference_metrics)
submission_path = None
if GENERATE_SUBMISSION and promising_candidate:
    if best_name not in test_probabilities:
        print('Candidate qualifies, but SAVE_TEST_PROBABILITIES=False. No submission generated.')
    else:
        original_sample = pd.read_csv(sample_path, dtype={'image_id': str}, keep_default_na=False)
        submission = build_submission(TEST_IDS, test_probabilities[best_name], original_sample)
        submission_path = OUTPUT_DIR / f'H3_{best_name}_{IMAGE_SIZE}_submission.csv'
        submission.to_csv(submission_path, index=False, mode='x')
        print('Saved candidate:', submission_path)
        display(submission.head())
else:
    print('No submission generated: disabled or selected H3 model did not beat the F0 reference ranking.')

summary = {'experiment': 'H3_frozen_backbone_comparison', 'run_id': RUN_ID,
           'reference_kind': reference_kind, 'reference_metrics': reference_metrics,
           'previously_reported_F0': REPORTED_F0,
           'reference_checkpoint': str(reference_path) if reference_path else None,
           'candidates': results, 'selected': best_result,
           'promising_candidate': promising_candidate,
           'submission_path': str(submission_path) if submission_path else None,
           'output_dir': str(OUTPUT_DIR), 'dataset_signature': DATA_SIGNATURE,
           'note': 'Validation comparison only; Kaggle effect is unknown.'}
(OUTPUT_DIR / 'H3_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('Send back H3_summary.json, H3_comparison.csv, H3_disagreements.csv, and H3_training_curves.png.')
print('Retain the complete run directory and feature caches in Drive for later fine-tuning/ensembling.')